In [ ]:
"""
Load: Download and parse a PDF document
Split: Chunk the document into smaller pieces
Embed & Store: Create embeddings and store in a vector database
Query: Retrieve relevant chunks and generate responses

What you'll learn:

Build a basic RAG pipeline from scratch using LangChain
Load, split, embed, and store PDF documents in a vector database (LanceDB)
Create a retrieval-augmented generation chain using LCEL
Query the system and get context-grounded answers
"""

In [ ]:
!pip install -q langchain langchain-community langchain-openai langchain-cohere langchain-text-splitters lancedb

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import LanceDB
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os
from langchain_cohere import CohereEmbeddings

In [ ]:
from google.colab import userdata
COHERE_API_KEY= userdata.get('COHERE')
OPENROUTER_API_KEY= userdata.get('OPENROUTER_API_KEY')

In [ ]:
# Load the Knowledge

loader = TextLoader("/content/wesla_rag_knowledge_base.md", encoding="utf-8")
pages = loader.load()


In [ ]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=200,
    length_function=len,
)

chunks = text_splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")


In [ ]:
# Create embeddings and vector store
embeddings = CohereEmbeddings(model="embed-multilingual-light-v3.0",cohere_api_key=COHERE_API_KEY)
vectorstore = LanceDB.from_documents(chunks, embeddings)

print(f"Created vector store with {len(chunks)} documents")


In [ ]:
# Create the RAG chain using LCEL (LangChain Expression Language)
llm = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=0,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:

{context}

Question: {question}"""
)

rag_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()
     }
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
# Ask questions
questions = [
    "إيه الباقات المتاحة للموبايل وأسعارها؟",
    "باقة وصلة 75 فيها كام وحدة وصلاحيتها قد إيه؟",
    "إزاي أجدد باقة الإنترنت؟",
    "إزاي أعرف رصيدي المتبقي؟",
    "الإنترنت عندي بطيء، أعمل إيه؟",
    "إيه إعدادات الـ APN؟",
    "إزاي أحول رصيد لرقم تاني؟",
    "هل الوحدات المتبقية بتترحل؟",
    "إزاي أقدم شكوى لخدمة العملاء؟",
    "مواعيد خدمة العملاء إيه؟",
    "الراوتر مش شغال، أتصرف إزاي؟",
    "إزاي أغير باسورد الواي فاي؟",
    "إيه باقات الإنترنت المنزلي؟",
    "شحنت رصيد ومظهرش عندي، أعمل إيه؟",
    "إزاي أوقف الخط لو الموبايل اتسرق؟",
    "هل فيه خدمة إنترنت فضائي؟",
    "مين مالك شركة وصلة؟"
]

In [ ]:

for q in questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)
